In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df=pd.read_csv("loan_approval_data.csv")
df=df.drop("Applicant_ID",axis=1)

catogru_columns=df.select_dtypes(include=["object"]).columns
numerical_columns =df.select_dtypes(include=["float"]).columns

from sklearn.impute import SimpleImputer
num_imp=SimpleImputer(strategy="mean")
df[numerical_columns]=num_imp.fit_transform(df[numerical_columns])

cat_imp=SimpleImputer(strategy="most_frequent")
df[catogru_columns]=cat_imp.fit_transform(df[catogru_columns])

from sklearn.preprocessing import OneHotEncoder,LabelEncoder
colms=["Marital_Status","Employment_Status","Loan_Purpose","Property_Area","Gender",
      "Employer_Category"]

# label encoding
le=LabelEncoder()
df["Education_Level"]=le.fit_transform(df["Education_Level"])
df["Loan_Approved"]=le.fit_transform(df["Loan_Approved"])

# OneHotCoding
ohe=OneHotEncoder(sparse_output=False,
                 handle_unknown='ignore',
                 drop='first')
encoded=ohe.fit_transform(df[colms])
encoded_df=pd.DataFrame(encoded,columns=ohe.get_feature_names_out(colms),index=df.index)

df=pd.concat([df.drop(columns=colms),encoded_df],axis=1)

x=df[[
    "Credit_Score",
    "Applicant_Income",
    "Loan_Amount",
    "DTI_Ratio",
    "Loan_Term"]]

y=df["Loan_Approved"]

x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42)

from sklearn.preprocessing import StandardScaler 
scaler=StandardScaler() 
x_train_scaled=scaler.fit_transform(x_train) 
x_test_scaled=scaler.transform(x_test)


# for vanie bais

from sklearn.naive_bayes import GaussianNB
model= GaussianNB()
model.fit(x_train_scaled,y_train)
y_pred=model.predict(x_test_scaled)

# evulate
from sklearn.metrics import (
accuracy_score,confusion_matrix,precision_score,recall_score,f1_score
)

acuuracy=accuracy_score(y_test,y_pred)
confusion = confusion_matrix(y_test,y_pred)
precsion=precision_score(y_test,y_pred)
recall=recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)

print("NAvie bayes")
print("accuracy : " ,acuuracy)
print("confusio : ",confusion)
print("precsion : " ,precsion)
print("recall   : ",recall)
print("f1 : ",f1)




            






NAvie bayes
accuracy :  0.885
confusio :  [[131   8]
 [ 15  46]]
precsion :  0.8518518518518519
recall   :  0.7540983606557377
f1 :  0.8


In [5]:
print(x.shape)

(1000, 5)


In [17]:
import pickle
pickle.dump(model,
            open("loan_model.pkl","wb"))
pickle.dump(scaler,
            open("scaler.pkl","wb"))


In [4]:
print(y.value_counts())

Loan_Approved
0    702
1    298
Name: count, dtype: int64


In [5]:
print(model.predict(x_test_scaled[:20]))

[0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 0 1]


In [6]:
print(x.columns)

Index(['Credit_Score', 'Applicant_Income', 'Loan_Amount', 'DTI_Ratio',
       'Loan_Term'],
      dtype='object')


In [9]:
print(model.predict_proba(x_train_scaled[:10]))

[[0.96809669 0.03190331]
 [0.94984393 0.05015607]
 [0.98354728 0.01645272]
 [0.56550532 0.43449468]
 [0.5626571  0.4373429 ]
 [0.46058232 0.53941768]
 [0.33755676 0.66244324]
 [0.35712902 0.64287098]
 [0.98935624 0.01064376]
 [0.99050463 0.00949537]]


In [10]:
print(y_test[0:10])

521    0
737    0
740    0
660    0
411    0
678    1
626    0
513    0
859    0
136    0
Name: Loan_Approved, dtype: int64


In [16]:
x=scaler.transform([[721,13000,12741,0.29,24]])
print(model.predict(x))
print(model.predict_proba(x))

[1]
[[0.20956981 0.79043019]]


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [12]:
approve=x[y==1]
print(approve.head())

IndexError: boolean index did not match indexed array along axis 0; size of axis is 1 but size of corresponding boolean axis is 1000

In [13]:
print(model.classes_)

[0 1]


In [15]:
print(df[df["Loan_Approved"]==1].head())

    Applicant_Income  Coapplicant_Income   Age  Dependents  Credit_Score  \
2             7390.0              2106.0  25.0         2.0         674.0   
4            13284.0              4223.0  31.0         2.0         721.0   
9            13363.0              2599.0  35.0         3.0         726.0   
10           18023.0              4033.0  52.0         2.0         688.0   
11           10322.0              8428.0  41.0         1.0         743.0   

    Existing_Loans  DTI_Ratio       Savings  Collateral_Value  Loan_Amount  \
2              4.0   0.200000  13844.000000            6908.0      27943.0   
4              1.0   0.290000   9386.000000           37629.0      12741.0   
9              1.0   0.347263   3022.000000           29693.0      22182.0   
10             1.0   0.130000  18244.000000           40004.0      10415.0   
11             4.0   0.230000   9940.452632           38333.0      12085.0   

    ...  Loan_Purpose_Education  Loan_Purpose_Home  Loan_Purpose_Personal 